# MXene Benchmark - Sequential Open-Weight Candidates

Runs gpt-oss-20b, Qwen3.6-35B-A3B, and gemma-4-12B-it **sequentially in one notebook**, sweeping each model's thinking-level variants while it's loaded, then explicitly freeing the model and its downloaded GGUF file before the next one loads - Kaggle's disk quota isn't big enough to hold multiple large checkpoints at once.

Uses **llama.cpp** (via `llama-cpp-python`) with unsloth's 4-bit-class GGUF quantizations, not vLLM: confirmed live that Kaggle assigns this kernel a P100 (compute capability 6.0), which is incompatible with every native quantization format the original vLLM design used (MXFP4 and FP8 both need Ampere+ 8.0; W4A16-marlin needs Volta+ 7.0). GGUF's k-quants run through llama.cpp's own portable CUDA kernels instead, sidestepping that hardware gate entirely.

Uses the exact same prompt template and JSON/validation logic as the local pipeline and the reference-panel runs (bundled via the `mxene-benchmark-subset` Kaggle Dataset attached to this notebook), so results are directly comparable via `benchmark.run_benchmark`.

**Known rough edges, noted honestly rather than hidden:**
- Each model exposes "thinking" differently, and these are prompt/message-level conventions rather than API parameters (llama.cpp's chat-completion layer doesn't have a documented equivalent to vLLM's `chat_template_kwargs`): gpt-oss-20b via a `Reasoning: low/medium/high` system message (documented OpenAI mechanism), Qwen3.6 via flipping the shared prompt's trailing `/no_think` to `/think` (its own chat-template convention), and gemma-4-12B-it (whose real mechanism is a `thinking_budget` token count, not named levels) via a plain-text reasoning-budget instruction appended to the prompt.
- Multi-GPU layer splitting (needed for the ~17.7GB Qwen file on a single 16GB GPU) relies on `llama-cpp-python`'s default split behavior across visible CUDA devices when `n_gpu_layers=-1` - not hands-on verified against Kaggle's specific T4x2 environment.

In [ ]:
import subprocess
import sys
import os

# Prefer a prebuilt CUDA wheel over compiling from source. Confirmed on a live run: this
# container reports CUDA 12.8, but abetlen's wheel index only publishes cu121/cu124 - trying
# the exact detected version resolves to nothing and pip silently falls back to a CPU-only
# PyPI wheel (a real risk, checked explicitly below via llama_supports_gpu_offload()). CUDA
# drivers are backward-compatible, so an older wheel tag should still run fine here.
import torch
print('Detected CUDA:', torch.version.cuda)

def try_wheel(cuda_tag):
    print(f'Trying prebuilt wheel tag {cuda_tag}...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'llama-cpp-python', '--force-reinstall',
         '--extra-index-url', f'https://abetlen.github.io/llama-cpp-python/whl/{cuda_tag}'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f'  pip install failed for {cuda_tag}')
        return False
    check = subprocess.run(
        [sys.executable, '-c', 'from llama_cpp import llama_cpp as _l; print(_l.llama_supports_gpu_offload())'],
        capture_output=True, text=True,
    )
    ok = check.stdout.strip() == 'True'
    print(f'  {cuda_tag}: GPU offload supported = {ok}')
    return ok

installed = any(try_wheel(tag) for tag in ('cu124', 'cu122', 'cu121'))

if not installed:
    print('No working prebuilt CUDA wheel found - falling back to a verbose source build, with an '
          'explicit CUDAToolkit stub-library path (fixes the "CUDA::cuda_driver ... find_package call '
          'is missing" CMake error seen on a live run, where the driver stub was not on CMake\'s default '
          'search path).')
    env = {
        **os.environ,
        'CMAKE_ARGS': '-DGGML_CUDA=on -DCUDAToolkit_ROOT=/usr/local/cuda '
                      '-DCMAKE_LIBRARY_PATH=/usr/local/cuda/lib64/stubs',
    }
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'llama-cpp-python', '--no-cache-dir', '--force-reinstall', '--verbose'],
        capture_output=True, text=True, env=env,
    )
    print(result.stdout[-8000:])
    print(result.stderr[-8000:])
    if result.returncode != 0:
        raise RuntimeError('Failed to install llama-cpp-python via prebuilt wheels and a source build - see output above')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)

import llama_cpp
from llama_cpp import llama_cpp as _llama_cpp_lib
print('llama_cpp installed OK, version:', getattr(llama_cpp, '__version__', 'unknown'))
print('GPU offload supported:', _llama_cpp_lib.llama_supports_gpu_offload())

In [ ]:
import gc
import json
import sys
import time
from pathlib import Path

import torch

OUTPUT_ROOT = Path('/kaggle/working')

# Kaggle's dataset mount path has changed across environments (classic flat
# /kaggle/input/<slug>/ vs newer /kaggle/input/datasets/<owner>/<slug>/ nesting) - rather
# than hardcode one, search for wherever json_utils.py actually landed. Also retry for up
# to 5 minutes since newly-created private datasets can take a bit to finish mounting.
BUNDLE_DIR = None
for attempt in range(10):
    matches = list(Path('/kaggle/input').rglob('json_utils.py'))
    if matches:
        BUNDLE_DIR = matches[0].parent
        break
    print(f'Waiting for dataset to mount ({attempt + 1}/10)... contents so far:', list(Path('/kaggle/input').rglob('*')))
    time.sleep(30)

if BUNDLE_DIR is None:
    raise FileNotFoundError('json_utils.py never appeared anywhere under /kaggle/input - check the dataset is attached correctly')

print('Found bundle at:', BUNDLE_DIR)
print('Contents of BUNDLE_DIR:', list(BUNDLE_DIR.iterdir()))
sys.path.append(str(BUNDLE_DIR))

from json_utils import extract_json_from_response
from validator import Validator

with open(BUNDLE_DIR / 'benchmark_subset.json', encoding='utf-8') as f:
    papers = json.load(f)

with open(BUNDLE_DIR / 'extraction_prompt.txt', encoding='utf-8') as f:
    PROMPT_TEMPLATE = f.read()

with open(BUNDLE_DIR / 'kaggle_candidates.json', encoding='utf-8') as f:
    KAGGLE_CANDIDATES = json.load(f)

print(f'Loaded {len(papers)} papers, {len(KAGGLE_CANDIDATES)} candidate models: {list(KAGGLE_CANDIDATES)}')

In [ ]:
def build_messages(model_key: str, thinking_param, prompt_text: str):
    """Returns messages - the per-model mechanism for controlling reasoning effort, see the
    caveats in the intro markdown cell above. All three are prompt/message-level conventions
    (not API parameters), since llama.cpp's chat-completion layer has no documented equivalent
    to vLLM's chat_template_kwargs."""
    if model_key == 'gpt-oss-20b':
        level = thinking_param  # 'low' | 'medium' | 'high'
        return [
            {'role': 'system', 'content': f'Reasoning: {level}'},
            {'role': 'user', 'content': prompt_text},
        ]

    if model_key == 'qwen3.6-35b-a3b':
        thinking_on = (thinking_param == 'thinking-on')
        text = prompt_text.replace('/no_think', '/think') if thinking_on else prompt_text
        return [{'role': 'user', 'content': text}]

    if model_key == 'gemma-4-12b-it':
        budget = thinking_param  # int token count (0 = disabled)
        if budget and budget > 0:
            directive = f'\n\n(Think through this carefully for up to {budget} tokens before giving your final answer.)'
        else:
            directive = '\n\n(Answer directly without extended step-by-step reasoning.)'
        return [{'role': 'user', 'content': prompt_text + directive}]

    return [{'role': 'user', 'content': prompt_text}]


def variant_items(thinking_variants):
    """Normalizes both shapes used in KAGGLE_CANDIDATES: a list of labels (label==param), or a
    dict of label->param (e.g. gemma's named levels -> token budgets)."""
    if isinstance(thinking_variants, dict):
        return list(thinking_variants.items())
    return [(v, v) for v in thinking_variants]

In [ ]:
import os
import traceback

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

validator = Validator()

# Grammar-constrained JSON generation: an earlier live run showed ~90% of gpt-oss-20b's
# free-form responses failing to parse (malformed JSON, or the model's Harmony-format
# reasoning channel leaking into the output). response_format with a schema uses llama.cpp's
# GBNF grammar support to constrain token selection so the model literally cannot emit
# invalid JSON or non-JSON content. This worked well for Qwen/Gemma, but regressed
# gpt-oss-20b to 100% schema-valid-but-empty extractions - forcing strict JSON from token 1
# removes the "thinking room" its Harmony format trains it to use before answering. See
# benchmark.config.KAGGLE_CANDIDATES's "use_json_schema" flag: False for gpt-oss-20b (relies
# on the Harmony-tag-stripping + greedy-regex parser in json_utils.py instead), True for
# Qwen/Gemma. The schema mirrors prompts/extraction_prompt.txt's structure.
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "materials": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "mxene_composition": {"type": "string"},
                    "composite_material": {"type": "string"},
                    "synthesis_method": {"type": "string"},
                    "fabrication_method": {"type": "string"},
                },
            },
        },
        "properties": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "property_type": {"type": "string"},
                    "value": {"type": "number"},
                    "unit": {"type": "string"},
                    "test_conditions": {"type": "string"},
                },
            },
        },
        "applications": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "application_type": {"type": "string"},
                    "metric": {"type": "string"},
                    "value": {"type": "number"},
                    "unit": {"type": "string"},
                    "notes": {"type": "string"},
                },
            },
        },
    },
    "required": ["materials", "properties", "applications"],
}
RESPONSE_FORMAT = {"type": "json_object", "schema": RESPONSE_SCHEMA}

for model_key, cfg in KAGGLE_CANDIDATES.items():
    # Each model is isolated in its own try/except/finally: a load or generation failure for
    # one model must not abort the whole notebook and lose every other model's results.
    model_path = None
    llm = None
    try:
        print(f"\n=== Downloading {model_key} ({cfg['gguf_repo']}/{cfg['gguf_filename']}) ===")
        model_path = hf_hub_download(repo_id=cfg['gguf_repo'], filename=cfg['gguf_filename'])

        print(f'=== Loading {model_key} from {model_path} ===')
        # Confirmed live: Qwen3.6-35B-A3B OOMs at n_gpu_layers=-1 on a single 16269 MiB P100 -
        # every available quant exceeds that VRAM budget. Retry with progressively more
        # CPU-offloaded layers rather than needing a pre-guessed number - MoE models like Qwen
        # only activate a few billion of their total params per token, so CPU-spilled expert
        # layers should be tolerably slow rather than catastrophic.
        gpu_layers_to_try = cfg.get('gpu_layers_to_try', [-1])
        last_error = None
        for n_gpu_layers in gpu_layers_to_try:
            try:
                print(f'  Trying n_gpu_layers={n_gpu_layers}...')
                llm = Llama(
                    model_path=model_path,
                    n_gpu_layers=n_gpu_layers,
                    n_ctx=8192,
                    n_batch=512,
                    verbose=True,   # keep the real llama.cpp diagnostic visible if a load fails
                )
                print(f'  Loaded successfully with n_gpu_layers={n_gpu_layers}')
                break
            except Exception as e:
                last_error = e
                print(f'  Failed with n_gpu_layers={n_gpu_layers}: {e}')
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        if llm is None:
            raise RuntimeError(
                f'Could not load {model_key} at any n_gpu_layers setting {gpu_layers_to_try}'
            ) from last_error

        response_format = RESPONSE_FORMAT if cfg.get('use_json_schema', True) else None

        for label, param in variant_items(cfg['thinking_variants']):
            run_key = f'{model_key}__{label}'
            print(f'--- {run_key} ---')

            results = []
            failed_count = 0
            t0 = time.time()

            for paper in papers:
                base_prompt = PROMPT_TEMPLATE.format(
                    title=paper.get('title', ''), abstract=paper.get('abstract', ''), conclusion=paper.get('conclusion', '')
                )
                messages = build_messages(model_key, param, base_prompt)

                response_text = None
                try:
                    response = llm.create_chat_completion(
                        messages=messages, temperature=0.1, max_tokens=4096, response_format=response_format
                    )
                    response_text = response['choices'][0]['message']['content']
                except Exception as e:
                    print(f'Generation error on "{paper.get("title", "Unknown")[:50]}": {e}')

                extracted = extract_json_from_response(response_text) if response_text else None

                paper_out = dict(paper)
                validated = validator.validate_extracted_data(
                    extracted or {'materials': [], 'properties': [], 'applications': []}
                )
                paper_out['extracted_data'] = validated
                paper_out['extraction_failed'] = extracted is None
                results.append(paper_out)

                if extracted is None:
                    failed_count += 1

            total_time_s = time.time() - t0

            stats = {
                'run_key': run_key,
                'gguf_repo': cfg['gguf_repo'],
                'gguf_filename': cfg['gguf_filename'],
                'thinking_variant': label,
                'total_papers': len(papers),
                'successful_extractions': len(papers) - failed_count,
                'failed_extractions': failed_count,
                'total_time_s': round(total_time_s, 2),
                'avg_time_per_paper_s': round(total_time_s / len(papers), 3) if papers else 0,
            }

            out_dir = OUTPUT_ROOT / run_key
            out_dir.mkdir(parents=True, exist_ok=True)
            with open(out_dir / 'extractions.json', 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            with open(out_dir / 'run_stats.json', 'w', encoding='utf-8') as f:
                json.dump(stats, f, indent=2)

            print(stats)

    except Exception as e:
        print(f'\n=== {model_key} FAILED - skipping to next model ===')
        print(f'{type(e).__name__}: {e}')
        traceback.print_exc()

    finally:
        # Free GPU memory and disk regardless of whether this model succeeded, so a failure
        # here doesn't also starve the next model of VRAM/disk.
        if llm is not None:
            del llm
            gc.collect()
        if model_path and os.path.exists(model_path):
            os.remove(model_path)
            print(f'Freed disk: removed {model_path}')

print('\nAll candidate models attempted (see per-model status above for any that failed).')